# Agent Testing Notebook

This notebook allows you to test individual agents in isolation with mock data.

**Purpose:** Test each agent's behavior without running the full multi-agent workflow.

**Agents Covered:**
1. Supervisor Agent
2. Reasoning Agent
3. Analyzer Agent
4. Parser Agent
5. Critic Agent
6. Full Workflow Simulation

## Section 1: Setup

Import all required modules and set up helper functions for testing.

In [ ]:
# Install dependencies if needed
# %pip install -r requirements.txt
# dbutils.library.restartPython()  # Uncomment if in Databricks

In [ ]:
import json
from pprint import pprint
from typing import Dict, Any

# Import all agents
from multiAgentSystem.agents.supervisor import supervisor_node
from multiAgentSystem.agents.reasoning import reasoning_node
from multiAgentSystem.agents.analyzer import analyzer_node
from multiAgentSystem.agents.parser import parser_node
from multiAgentSystem.agents.critic import critic_node

# Import configuration
from multiAgentSystem.config import (
    configure_agent_llms,
    show_llm_configuration,
    MAX_OUTER_ITERATIONS,
    MAX_ANALYZE_PARSE_LOOPS
)

print("✓ Imports successful")
print(f"Max Outer Iterations: {MAX_OUTER_ITERATIONS}")
print(f"Max Analyze-Parse Loops: {MAX_ANALYZE_PARSE_LOOPS}")

In [ ]:
# Helper function to display results nicely
def display_result(title: str, result: Dict[str, Any]):
    """Display agent output in a readable format."""
    print("=" * 80)
    print(f"  {title}")
    print("=" * 80)
    for key, value in result.items():
        if isinstance(value, (dict, list)) and len(str(value)) > 200:
            print(f"\n{key}:")
            pprint(value, width=80, depth=2)
        else:
            print(f"\n{key}: {value}")
    print("=" * 80)


def create_mock_evidence_map():
    """Create mock evidence map for testing."""
    return {
        "OutOfMemoryError: Java heap space": {
            "count": 15,
            "timestamps": ["25/10/08 06:23:27", "25/10/08 06:23:35", "25/10/08 06:23:42"],
            "files": ["/Volumes/logs/executor-1.log", "/Volumes/logs/executor-2.log"],
            "sample_lines": [
                "25/10/08 06:23:27 ERROR Executor: OutOfMemoryError: Java heap space at shuffle",
                "25/10/08 06:23:35 ERROR Executor: OutOfMemoryError: Java heap space at collect"
            ]
        },
        "Executor lost failure": {
            "count": 3,
            "timestamps": ["25/10/08 06:24:01"],
            "files": ["/Volumes/logs/driver.log"],
            "sample_lines": [
                "25/10/08 06:24:01 WARN TaskSetManager: Lost executor * on *: Container killed by YARN"
            ]
        }
    }

print("✓ Helper functions defined")

In [ ]:
# Optional: Configure different LLMs for different agents
# Uncomment and modify as needed

# configure_agent_llms({
#     "reasoning": "databricks-claude-sonnet-4-5",
#     "supervisor": "databricks-claude-opus",
#     "analyzer": "databricks-claude-haiku",
#     "parser": "databricks-claude-haiku",
#     "critic": "databricks-claude-sonnet-4-5"
# })

# Show current LLM configuration
show_llm_configuration()

## Section 2: Supervisor Agent Testing

The Supervisor agent decides which agent to invoke next based on the current state.

**Key Inputs:**
- `iteration`: Current iteration count
- `last_status`: Status from reasoning ("continue", "summarized", or "")
- `confidence`: Current confidence score (0.0-1.0)
- `critic_approved`: Whether critic has approved the draft

**Expected Output:**
- `next_action`: "reasoning", "critic", or "end"
- `supervisor_rationale`: Explanation of decision
- `iteration`: Updated iteration count

In [ ]:
# Test Case 1: Initial state (iteration 0)
test_state_supervisor_1 = {
    "iteration": 0,
    "last_status": "",
    "confidence": 0.0,
    "critic_approved": False,
    "draft": {},
    "evidence_summary": "",
    "critique": ""
}

result_1 = supervisor_node(test_state_supervisor_1)
display_result("Supervisor Test 1: Initial State", result_1)

# Expected: next_action = "reasoning" (always starts with reasoning)

In [ ]:
# Test Case 2: After reasoning completed (needs critic)
test_state_supervisor_2 = {
    "iteration": 1,
    "last_status": "summarized",
    "confidence": 0.65,
    "critic_approved": False,
    "draft": {
        "problem": "Job failed due to OOM",
        "rca": "[PROVEN] Executors ran out of memory",
        "mitigation": "Increase executor memory"
    },
    "evidence_summary": "Found 15 OOM errors",
    "critique": ""
}

result_2 = supervisor_node(test_state_supervisor_2)
display_result("Supervisor Test 2: After Summarization", result_2)

# Expected: next_action = "critic" (needs validation)

In [ ]:
# Test Case 3: Critic approved with high confidence
test_state_supervisor_3 = {
    "iteration": 2,
    "last_status": "summarized",
    "confidence": 0.85,
    "critic_approved": True,
    "draft": {
        "problem": "Job failed due to OOM",
        "rca": "[PROVEN] Executors ran out of memory",
        "mitigation": "Increase executor memory"
    },
    "evidence_summary": "Found 15 OOM errors",
    "critique": "Analysis is well-supported"
}

result_3 = supervisor_node(test_state_supervisor_3)
display_result("Supervisor Test 3: Approved + High Confidence", result_3)

# Expected: next_action = "end" (confidence >= 0.75 and approved)

## Section 3: Reasoning Agent Testing

The Reasoning agent assesses evidence sufficiency and generates hypotheses or creates the final RCA draft.

**Key Inputs:**
- `user_context`: Problem description
- `hypotheses`: Current hypotheses
- `evidence_summary`: Collected evidence
- `keywords`: Keywords used
- `iteration`: Current iteration

**Expected Output:**
- If insufficient evidence: `next_action="analyzer"`, updated `hypotheses`
- If sufficient: `draft` with problem/rca/mitigation, `confidence`, `last_status="summarized"`

In [ ]:
# Test Case 1: Initial assessment (no evidence)
test_state_reasoning_1 = {
    "user_context": "Spark job failed with executor losses during shuffle phase",
    "logs_path": "/Volumes/test/logs",
    "hypotheses": [],
    "evidence_map": {},
    "evidence_summary": "",
    "keywords": [],
    "iteration": 0,
    "analyze_parse_loops": 0
}

result_reasoning_1 = reasoning_node(test_state_reasoning_1)
display_result("Reasoning Test 1: No Evidence", result_reasoning_1)

# Expected: next_action = "analyzer" (needs more evidence)
# Expected: hypotheses populated with initial hypotheses

In [ ]:
# Test Case 2: With evidence (should generate draft)
mock_evidence_summary = """=== Evidence Summary (2 unique error patterns) ===

[1] Error Pattern: OutOfMemoryError: Java heap space
    Occurrences: 15
    Timestamps: 25/10/08 06:23:27, 25/10/08 06:23:35, 25/10/08 06:23:42 ... (+12 more)
    Files: /Volumes/logs/executor-1.log, /Volumes/logs/executor-2.log
    Sample: 25/10/08 06:23:27 ERROR Executor: OutOfMemoryError: Java heap space at shuffle...

[2] Error Pattern: Executor lost failure
    Occurrences: 3
    Timestamps: 25/10/08 06:24:01
    Files: /Volumes/logs/driver.log
    Sample: 25/10/08 06:24:01 WARN TaskSetManager: Lost executor * on *: Container killed by YARN...
"""

test_state_reasoning_2 = {
    "user_context": "Spark job failed with executor losses during shuffle phase",
    "logs_path": "/Volumes/test/logs",
    "hypotheses": [
        "Memory exhaustion during shuffle",
        "Insufficient executor memory allocation"
    ],
    "evidence_map": create_mock_evidence_map(),
    "evidence_summary": mock_evidence_summary,
    "keywords": ["OutOfMemoryError", "executor lost", "shuffle"],
    "iteration": 1,
    "analyze_parse_loops": 2
}

result_reasoning_2 = reasoning_node(test_state_reasoning_2)
display_result("Reasoning Test 2: With Evidence", result_reasoning_2)

# Expected: draft populated with problem, rca, mitigation
# Expected: confidence calculated
# Expected: last_status = "summarized"

## Section 4: Analyzer Agent Testing

The Analyzer agent converts hypotheses into search keywords.

**Key Inputs:**
- `user_context`: Problem description
- `hypotheses`: Current hypotheses
- `keywords`: Existing keywords
- `last_logs_chunk`: Latest log search results

**Expected Output:**
- `keywords`: Updated keyword list (DEFAULT_KEYWORDS + LLM-generated)
- `last_generated_keywords`: Keywords from this run
- `analyze_parse_loops`: Incremented counter

In [ ]:
# Test Case 1: Initial keyword generation
test_state_analyzer_1 = {
    "user_context": "Job failed with OOM errors during shuffle",
    "hypotheses": [
        "Memory exhaustion during shuffle",
        "Data skew causing memory pressure"
    ],
    "keywords": [],
    "last_logs_chunk": "",
    "analyze_parse_loops": 0
}

result_analyzer_1 = analyzer_node(test_state_analyzer_1)
display_result("Analyzer Test 1: Initial Keywords", result_analyzer_1)

# Expected: keywords include DEFAULT_KEYWORDS + OOM-related keywords
# Expected: analyze_parse_loops = 1

In [ ]:
# Test Case 2: Refined keyword generation (with previous results)
test_state_analyzer_2 = {
    "user_context": "Job failed with GC overhead errors",
    "hypotheses": [
        "GC pauses too long",
        "Stop-the-world events causing timeouts"
    ],
    "keywords": ["ERROR", "Exception", "OutOfMemoryError"],
    "last_logs_chunk": "Found multiple GC overhead errors in executor logs",
    "analyze_parse_loops": 1
}

result_analyzer_2 = analyzer_node(test_state_analyzer_2)
display_result("Analyzer Test 2: Refined Keywords", result_analyzer_2)

# Expected: keywords include GC-specific terms
# Expected: analyze_parse_loops = 2

## Section 5: Parser Agent Testing

The Parser agent searches logs using keywords and updates the evidence map.

**Note:** Parser requires actual log files or will return empty results.
For testing, we'll use mock paths (results will be empty but you'll see the flow).

**Key Inputs:**
- `logs_path`: Path to logs
- `last_generated_keywords`: Keywords to search
- `evidence_map`: Current evidence map

**Expected Output:**
- `evidence_map`: Updated with search results
- `evidence_summary`: Formatted summary
- `last_logs_chunk`: Raw search results

In [ ]:
# Test Case: Parser with keywords
# Note: This will not find actual logs (mock path), but shows the structure
test_state_parser = {
    "logs_path": "/Volumes/test/mock-logs/",  # Mock path
    "last_generated_keywords": [
        "OutOfMemoryError",
        "executor lost",
        "GC overhead"
    ],
    "keywords": ["ERROR", "Exception"],
    "evidence_map": {},
    "evidence": []
}

result_parser = parser_node(test_state_parser)
display_result("Parser Test: Log Search", result_parser)

# Expected: evidence_map initialized (may be empty if no logs found)
# Expected: evidence_summary generated
# Expected: last_logs_chunk with search results or error message

In [ ]:
# Test Case: Parser with pre-populated evidence map
test_state_parser_2 = {
    "logs_path": "/Volumes/test/mock-logs/",
    "last_generated_keywords": ["Container killed"],
    "keywords": ["ERROR"],
    "evidence_map": create_mock_evidence_map(),  # Pre-populate
    "evidence": []
}

result_parser_2 = parser_node(test_state_parser_2)
display_result("Parser Test 2: With Existing Evidence", result_parser_2)

# Expected: evidence_map retains previous entries
# Expected: evidence_summary includes all patterns

## Section 6: Critic Agent Testing

The Critic agent validates the draft RCA against collected evidence.

**Key Inputs:**
- `draft`: Dict with problem, rca, mitigation
- `evidence_summary`: Available evidence

**Expected Output:**
- `critic_approved`: Boolean (approve/reject)
- `critique`: Feedback string
- `confidence`: Adjusted confidence score

In [ ]:
# Test Case 1: Well-supported draft
test_state_critic_1 = {
    "draft": {
        "problem": "Spark job failed due to executor memory exhaustion during shuffle phase",
        "rca": """1. [PROVEN] Job failed at stage 3 during shuffle (Evidence: 15 OOM errors in logs)
2. [PROVEN] Executors ran out of heap memory (Evidence: OutOfMemoryError: Java heap space)
3. [PROVEN] Containers were killed by YARN (Evidence: Container killed messages)
4. [INFERRED] Memory configuration was insufficient for data volume""",
        "mitigation": """1. Increase executor memory from 4GB to 8GB (spark.executor.memory)
2. Enable dynamic allocation (spark.dynamicAllocation.enabled=true)
3. Tune memory fraction (spark.memory.fraction=0.8)"""
    },
    "evidence_summary": mock_evidence_summary,
    "confidence": 0.75
}

result_critic_1 = critic_node(test_state_critic_1)
display_result("Critic Test 1: Well-Supported Draft", result_critic_1)

# Expected: critic_approved = True (or maybe False if LLM is strict)
# Expected: critique with validation details
# Expected: confidence may be adjusted slightly

In [ ]:
# Test Case 2: Draft with unsupported claims
test_state_critic_2 = {
    "draft": {
        "problem": "Job failed due to network issues",
        "rca": """1. [PROVEN] Network latency caused timeouts
2. [PROVEN] AWS experienced outage in us-west-2
3. [PROVEN] Shuffle service was unreachable""",
        "mitigation": "Use different availability zone"
    },
    "evidence_summary": mock_evidence_summary,  # Only has OOM evidence
    "confidence": 0.90
}

result_critic_2 = critic_node(test_state_critic_2)
display_result("Critic Test 2: Unsupported Claims", result_critic_2)

# Expected: critic_approved = False (claims don't match evidence)
# Expected: critique lists missing evidence
# Expected: confidence lowered significantly

---

## Summary

This notebook demonstrated how each agent works in isolation:

1. **Supervisor**: Routes to appropriate agent based on state
2. **Reasoning**: Assesses evidence and generates hypotheses or draft
3. **Analyzer**: Converts hypotheses to search keywords
4. **Parser**: Searches logs and updates evidence map
5. **Critic**: Validates draft against evidence

### Key Takeaways:

- Each agent has specific inputs/outputs
- State flows between agents via the AgentState dict
- Evidence map provides smart deduplication
- Agents use LLMs configured per agent type

### Next Steps:

1. Run tests with your actual log paths
2. Experiment with different LLM endpoints per agent
3. Modify mock data to match your use cases
4. Use insights to tune prompts or configuration